# Glossolalia Dial — Micro-Spike

Glossolalia Dial — fine-tune an F5-TTS LoRA so a single control token (`tongues zero..four`) grades the intelligibility of the same input sentence in the same voice, from clean speech to phonotactically-valid English-native glossolalia. The fine-tune is the load-bearing trick: no prompt, no DSP plugin, and no shipped product (closed or open) does this — verified by three adversarial research workflows (originality landscape, Cocteau Twins frame audit, glossolalia-converter prior-art sweep).

**Gates that must ALL pass before we ship:**
- Whisper-WER rises monotonically across levels 0..4, Spearman ≥ +0.80 (positive sign — WER *rises* as dial rises).
- Resemblyzer cosine vs the level-0 reference stays ≥ 0.85 (voice preserved).
- Perceptual gate: hand-listen `dial=2` wavs — must be a partial dissolution, not bimodal collapse.

**Recommended order:** run the **200-clip micro-spike** (Cell 3 default, ~17 min on A100) first to confirm the LoRA learns the control token at all. If the micro-spike's Cell-7 verdict is PARTIAL or PASS, scale to the full 7,500-clip run before committing to the long training.

Runtime: A100/H100 ZeroGPU strongly recommended (F5-TTS ~5-7s per generated clip).

In [1]:
# Cell 1 — install F5-TTS + validation deps + repo scripts
!nvidia-smi -L
!apt-get -qq install -y ffmpeg >/dev/null
!pip install -q f5-tts g2p_en jiwer openai-whisper resemblyzer huggingface_hub pedalboard soundfile librosa scipy numpy peft >/dev/null
!pip install -q -U "torchao>=0.16.0" >/dev/null  # peft >=0.14 requires torchao>=0.16 to import; Colab ships 0.10
import nltk
for res in ('averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger', 'cmudict'):
    try: nltk.download(res, quiet=True)
    except Exception: pass
# Bring in the repo scripts. If you renamed the repo, adjust REPO_URL.
REPO_URL = 'https://github.com/akshan-main/glossolalia.git'
import os
if not os.path.isdir('/content/repo'):
    !git clone -q $REPO_URL /content/repo || echo 'clone failed - upload scripts/ manually via Files panel'
%cd /content/repo
!ls scripts/ | head -20
print('install done')

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-ab966f59-814f-63b9-35b2-b44c17c104d5)
/content/repo
build_coherence_dataset.py
build_phoneme_lm.py
corrupt_phonemes.py
evaluate_coherence_dial.py
fetch_data_inputs.py
generate_coherence_data.py
__init__.py
post_fx.py
push_data_inputs.py
sweep_dial.py
install done


In [2]:
# Cell 2 — pull data inputs (sentences + voice refs + phoneme LM) from HF
from huggingface_hub import snapshot_download
import shutil, os
REPO = 'akshan-main/glossolalia-inputs'   # populated by scripts/push_data_inputs.py before running this notebook
p = snapshot_download(repo_id=REPO, repo_type='dataset', local_dir='/content/repo/data_pull')
# Stage into the layout the scripts expect (data/sentences.txt, data/voices/v*.wav, data/phoneme_lm.npz)
os.makedirs('/content/repo/data', exist_ok=True)
os.makedirs('/content/repo/data/voices', exist_ok=True)
for f in ['sentences.txt', 'phoneme_lm.npz', 'cmudict.dict']:
    src = os.path.join(p, f)
    if os.path.exists(src): shutil.copy(src, f'/content/repo/data/{f}')
voices_dir = os.path.join(p, 'voices')
if os.path.isdir(voices_dir):
    for f in os.listdir(voices_dir):
        shutil.copy(os.path.join(voices_dir, f), f'/content/repo/data/voices/{f}')
print('  sentences:', len(open('/content/repo/data/sentences.txt').readlines()))
print('  voices   :', sorted(os.listdir('/content/repo/data/voices')))
assert os.path.exists('/content/repo/data/phoneme_lm.npz'), 'phoneme_lm.npz missing'

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

  sentences: 500
  voices   : ['v1.txt', 'v1.wav', 'v2.txt', 'v2.wav', 'v3.txt', 'v3.wav']


In [3]:
# Cell 3 — generate the training corpus on GPU (F5-TTS base, no LoRA yet)
# Scales:
#   micro-spike   ->  40 sentences  x 5 levels x 1 voice =   200 clips  (~17 min on A100)
#   single-voice  -> 500 sentences  x 5 levels x 1 voice = 2,500 clips  (~3-4 h)
#   full          -> 500 sentences  x 5 levels x 3 voices = 7,500 clips (~10-12 h)
# Recommended: run the 200-clip micro-spike FIRST. Train a tiny LoRA on it (Cell 5 with --epochs 1)
# and read Cell 7's verdict. If Spearman > 0.5, scale up. If 0, the corruption procedure or token
# format needs fixing BEFORE you spend 10 hours.
!python scripts/generate_coherence_data.py \
  --sentences data/sentences.txt \
  --voice v1:data/voices/v1.wav:data/voices/v1.txt \
  --lm data/phoneme_lm.npz \
  --out data/coherence \
  --max-sentences 40 \
  --levels 5 \
  --input-mode pseudo \
  --resume
# To scale: add `--voice v2:data/voices/v2.wav:data/voices/v2.txt --voice v3:data/voices/v3.wav:data/voices/v3.txt`
# and bump `--max-sentences 500`.

Generating 200 clips: 40 sentences x 1 voices x 5 levels
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Co

In [4]:
# Cell 4 — build the F5-TTS finetune dataset (CSV + JSONL + symlinked wavs)
!python scripts/build_coherence_dataset.py --data data/coherence --out data/coherence_ds
!head -3 data/coherence_ds/metadata.csv
!wc -l data/coherence_ds/metadata.csv

wrote 200 rows (0 missing) -> data/coherence_ds
  - data/coherence_ds/metadata.csv
  - data/coherence_ds/metadata.jsonl
  - data/coherence_ds/wavs (symlinks)
audio_path|text
wavs/clip_00000_v1_lv0.wav|[The moon] I gazed with a kind of wonder. | tongues zero
wavs/clip_00001_v1_lv1.wav|[The moon] I gazed with a kind of wonder. | tongues one
201 data/coherence_ds/metadata.csv


In [8]:
import os
print('=== /content/repo/data/coherence (Cell 3 output) ===')
print('  exists:', os.path.exists('/content/repo/data/coherence'))
!ls /content/repo/data/coherence/ 2>/dev/null
!echo "  manifest.jsonl lines:" $(wc -l < /content/repo/data/coherence/manifest.jsonl 2>/dev/null || echo 0)
!echo "  wavs count:" $(ls /content/repo/data/coherence/wavs 2>/dev/null | wc -l)
!ls /content/repo/data/coherence/wavs 2>/dev/null | head -3

print()
print('=== /content/repo/data/coherence_ds (Cell 4 output) ===')
print('  exists:', os.path.exists('/content/repo/data/coherence_ds'))
!ls /content/repo/data/coherence_ds/ 2>/dev/null
!echo "  metadata.csv lines:" $(wc -l < /content/repo/data/coherence_ds/metadata.csv 2>/dev/null || echo 0)
!head -3 /content/repo/data/coherence_ds/metadata.csv 2>/dev/null
!echo "  wavs count:" $(ls /content/repo/data/coherence_ds/wavs 2>/dev/null | wc -l)
!ls /content/repo/data/coherence_ds/wavs 2>/dev/null | head -3

print()
print('=== sample wav existence + duration check ===')
import csv as _csv
csvp = '/content/repo/data/coherence_ds/metadata.csv'
if os.path.exists(csvp):
    with open(csvp) as f:
        f.readline()
        for i, line in enumerate(f):
            if i >= 3: break
            rel = line.split('|', 1)[0].strip()
            full = os.path.join('/content/repo/data/coherence_ds', rel)
            print(f'  {rel}  exists={os.path.exists(full)}', end='')
            if os.path.exists(full):
                import torchaudio
                try:
                    info = torchaudio.info(full)
                    print(f'  dur={info.num_frames/info.sample_rate:.2f}s')
                except Exception as e:
                    print(f'  torchaudio.info failed: {e}')
            else:
                print()


=== /content/repo/data/coherence (Cell 3 output) ===
  exists: True
clip_00000_v1_lv0.json	clip_00067_v1_lv2.json	clip_00134_v1_lv4.json
clip_00000_v1_lv0.wav	clip_00067_v1_lv2.wav	clip_00134_v1_lv4.wav
clip_00001_v1_lv1.json	clip_00068_v1_lv3.json	clip_00135_v1_lv0.json
clip_00001_v1_lv1.wav	clip_00068_v1_lv3.wav	clip_00135_v1_lv0.wav
clip_00002_v1_lv2.json	clip_00069_v1_lv4.json	clip_00136_v1_lv1.json
clip_00002_v1_lv2.wav	clip_00069_v1_lv4.wav	clip_00136_v1_lv1.wav
clip_00003_v1_lv3.json	clip_00070_v1_lv0.json	clip_00137_v1_lv2.json
clip_00003_v1_lv3.wav	clip_00070_v1_lv0.wav	clip_00137_v1_lv2.wav
clip_00004_v1_lv4.json	clip_00071_v1_lv1.json	clip_00138_v1_lv3.json
clip_00004_v1_lv4.wav	clip_00071_v1_lv1.wav	clip_00138_v1_lv3.wav
clip_00005_v1_lv0.json	clip_00072_v1_lv2.json	clip_00139_v1_lv4.json
clip_00005_v1_lv0.wav	clip_00072_v1_lv2.wav	clip_00139_v1_lv4.wav
clip_00006_v1_lv1.json	clip_00073_v1_lv3.json	clip_00140_v1_lv0.json
clip_00006_v1_lv1.wav	clip_00073_v1_lv3.wav	clip_0014

In [ ]:
import soundfile as _sf
from pathlib import Path as _Path

class CoherenceDS(Dataset):
    def __init__(self, csv_path, mel_kwargs, max_dur=15.0):
        self.rows = []
        base = _Path(csv_path).parent
        with open(csv_path) as f:
            f.readline()  # skip header
            for line in f:
                line = line.rstrip('\n')
                if not line: continue
                parts = line.split('|', 1)
                if len(parts) < 2: continue
                rel_path, text = parts[0].strip(), parts[1].strip()
                wav_p = base / rel_path
                try:
                    info = _sf.info(str(wav_p))
                    dur = info.frames / info.samplerate
                    if dur > max_dur: continue
                except Exception:
                    continue
                self.rows.append((str(wav_p), text))
        from torchaudio.transforms import MelSpectrogram
        self.mel = MelSpectrogram(
            sample_rate=mel_kwargs['target_sample_rate'],
            n_fft=mel_kwargs['n_fft'], hop_length=mel_kwargs['hop_length'],
            win_length=mel_kwargs['win_length'], n_mels=mel_kwargs['n_mel_channels'],
            power=1, center=True, normalized=False)
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        wav_p, text = self.rows[i]
        y, sr = torchaudio.load(wav_p)
        if sr != 24000:
            y = torchaudio.functional.resample(y, sr, 24000)
        if y.shape[0] > 1: y = y.mean(0, keepdim=True)
        m = self.mel(y).squeeze(0).clamp(min=1e-5).log()
        return {'mel': m.transpose(0,1), 'text': text, 'mel_len': m.shape[-1]}

ds = CoherenceDS('/content/repo/data/coherence_ds/metadata.csv', mel_kwargs)
dl = DataLoader(ds, batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2, drop_last=True)
print(f'  dataset rows: {len(ds)}, batches/epoch: {len(dl)}')
assert len(ds) > 0, 'dataset still empty — paste me the output and stop'

import time
trainable = [p for p in cfm.parameters() if p.requires_grad]
opt = torch.optim.AdamW(trainable, lr=LR, betas=(0.9, 0.99), weight_decay=0.0)
scaler = torch.amp.GradScaler('cuda')
warmup = 30
cfm.train()
step = 0; t0 = time.time()
for ep in range(EPOCHS):
    for batch in dl:
        mel = batch['mel'].to(device)
        mel_lens = batch['mel_lens'].to(device)
        text_in = batch['text']
        lr_now = LR * min(1.0, (step + 1) / warmup)
        for g in opt.param_groups: g['lr'] = lr_now
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            loss, _, _ = cfm(mel, text=text_in, lens=mel_lens)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        scaler.step(opt); scaler.update()
        if step % 10 == 0:
            print(f'  ep{ep} step{step:4d} loss={loss.item():.4f} lr={lr_now:.2e} elapsed={time.time()-t0:.0f}s')
        step += 1

cfm.transformer.save_pretrained(ADAPTER_DIR, safe_serialization=True)
print(f'\nsaved adapter -> {ADAPTER_DIR}')
print('  files:', __import__('os').listdir(ADAPTER_DIR))

import sys
sys.path.insert(0, '/content/repo')
import patches
from f5_tts.api import F5TTS as _F5TTS_check
print('F5TTS.load_lora installed:', hasattr(_F5TTS_check, 'load_lora'))


In [ ]:
# Cell 6 — sweep the dial across (sentence, voice, level, seed)
# Cell 5 writes a PEFT adapter directory (not a single .safetensors blob).
# Point sweep_dial.py at the dir. patches/ is auto-imported by sweep_dial.py itself.
import os, sys
sys.path.insert(0, '/content/repo')
import patches  # reinstall load_lora in case kernel was restarted
ADAPTER_DIR = '/content/repo/ckpts/unlang_lora/lora_last'
assert os.path.isfile(f'{ADAPTER_DIR}/adapter_model.safetensors'), 'Cell 5 must finish first'
print('using', ADAPTER_DIR)
!python scripts/sweep_dial.py \
  --lora $ADAPTER_DIR \
  --voices v1:data/voices/v1.wav:data/voices/v1.txt \
  --out sweep \
  --levels 5 \
  --seeds 42,123,7 \
  --max-sentences 10
# Multi-voice scale-up after micro-spike:
# --voices v1:data/voices/v1.wav:data/voices/v1.txt,v2:data/voices/v2.wav:data/voices/v2.txt,v3:data/voices/v3.wav:data/voices/v3.txt


In [ ]:
# Cell 7 — validate: Whisper WER (monotonic + Spearman ≥ +0.80) + Resemblyzer cosine (≥ 0.85)
!python scripts/evaluate_coherence_dial.py \
  --manifest sweep/sweep_manifest.json \
  --out sweep/eval_report.json \
  --whisper base.en \
  --spearman-gate 0.80 \
  --cosine-gate 0.85

In [ ]:
# Cell 8 — download dial=0/2/4 wavs for the human perceptual gate
from google.colab import files
import glob
picks = []
for lv in (0, 2, 4):
    g = sorted(glob.glob(f'sweep/v1_lv{lv}_s42_*.wav'))
    if g: picks.append(g[0])
for p in picks:
    print(p); files.download(p)

## Reading the result

- **All gates pass** (Spearman ≥ +0.80, min cosine ≥ 0.85, dial=2 perceptually partial) → push the LoRA + sweep wavs, build the toy.
- **Micro-spike Spearman 0** → the LoRA isn't learning the token. Check: is `tongues four` being split into chars (look at training logs)? Did F5-TTS swallow the `|` separator? Try the community LoRA fork.
- **Micro-spike Spearman 0.3-0.6** → real signal, just under-trained. Scale to full 7,500-clip run + 5 epochs.
- **WER non-monotonic** at full scale → tighten Markov LM to a vowel-heavy / sonorant-consonant palette and retrain.
- **Voice cosine < 0.85 at high levels** → lower rank to 8 + restrict target_modules to text-cross-attn only.
- **Dial=2 bimodal** → flatten the p-schedule (e.g., `[0, 0.20, 0.45, 0.70, 0.95]`) and curriculum-finetune on level-2-weighted mini-batches.

Push the trained LoRA:
```python
from huggingface_hub import HfApi
HfApi().upload_folder(folder_path='/content/repo/ckpts/unlang_lora', repo_id='akshan-main/glossolalia-dial-lora', repo_type='model')
```